In [1]:
from datasets import load_dataset
import json

dataset_treino = load_dataset("json", data_files="treino_formatado.jsonl", split="train")
print(dataset_treino)

with open("teste.json", encoding="utf-8") as f:
    teste = json.load(f)
print(f"Exemplos de teste: {len(teste)}")

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['messages'],
    num_rows: 80
})
Exemplos de teste: 20


#Baseline: Modelo sem fine-tuning no conjunto de teste

In [2]:
!pip install transformers peft bitsandbytes trl accelerate -q

In [3]:
import torch

print("CUDA disponível:", torch.cuda.is_available())

CUDA disponível: True


In [4]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
)
model.eval()

SYSTEM_PROMPT = (
    "Você é um sistema de extração de informações de anúncios de peças automotivas. "
    "Dado o texto de um anúncio, extraia os seguintes campos em formato JSON: "
    "categoria, marca_veiculo, modelo_veiculo, ano_compativel, posicao. "
    "Se um campo não estiver presente no texto, retorne null para ele."
)

def gerar_extracao(texto: str) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": texto},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        saida = model.generate(**inputs, max_new_tokens=100, do_sample=False)

    texto_gerado = tokenizer.decode(saida[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return texto_gerado

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

# Rodar o modelo sem fine-tuning nos 20 exemplos de teste

In [5]:
import json
import re

with open('teste.json', encoding='utf-8') as f:
    teste = json.load(f)

def extrair_json_da_resposta(texto_gerado: str):
    match = re.search(r'\{.*\}', texto_gerado, re.DOTALL)
    if not match:
        return None
    try:
        return json.loads(match.group(0))
    except json.JSONDecodeError:
        return None

resultados_baseline = []
for item in teste:
    saida_bruta = gerar_extracao(item["texto"])
    extraido = extrair_json_da_resposta(saida_bruta)
    resultados_baseline.append({
        "texto": item["texto"],
        "esperado": item["extracao"],
        "gerado": extraido,
        "saida_bruta": saida_bruta,
    })

with open('resultados_baseline.json', 'w', encoding='utf-8') as f:
    json.dump(resultados_baseline, f, ensure_ascii=False, indent=2)

print("Baseline concluído.")

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Baseline concluído.


#Fine-tuning com LoRA + QLoRA
Recarregando o modelo do zero antes de aplicar o adaptador pra garantir sessão limpa

In [6]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
)

#Essencial, sem esse comando o adaptador corrompe silenciosamente
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

trainable params: 4,358,144 || all params: 1,548,072,448 || trainable%: 0.2815


#Treinando

In [7]:
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

dataset_treino = load_dataset("json", data_files="treino_formatado.jsonl", split="train")

training_args = SFTConfig(
    output_dir="./autopecas_extrator_lora",
    num_train_epochs=6,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    logging_steps=2,
    max_length=512,
    report_to="none",
    save_strategy="no",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset_treino,
)

trainer.train()

Tokenizing train dataset:   0%|          | 0/80 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/80 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/80 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
2,2.425708
4,2.148959
6,1.870349
8,1.672661
10,1.450257
12,1.235264
14,1.092484
16,0.852920
18,0.864156
20,0.741292


TrainOutput(global_step=60, training_loss=0.8454429884751637, metrics={'train_runtime': 401.6949, 'train_samples_per_second': 1.195, 'train_steps_per_second': 0.149, 'total_flos': 643045453578240.0, 'train_loss': 0.8454429884751637, 'epoch': 6.0})

reativar o cache e colocar em modo avaliação

In [8]:
model.eval()
model.config.use_cache = True

In [9]:
import json
import re

with open('teste.json', encoding='utf-8') as f:
    teste = json.load(f)

def extrair_json_da_resposta(texto_gerado: str):
    match = re.search(r'\{.*\}', texto_gerado, re.DOTALL)
    if not match:
        return None
    try:
        return json.loads(match.group(0))
    except json.JSONDecodeError:
        return None

resultados_finetuned = []
for item in teste:
    saida_bruta = gerar_extracao(item["texto"])
    extraido = extrair_json_da_resposta(saida_bruta)
    resultados_finetuned.append({
        "texto": item["texto"],
        "esperado": item["extracao"],
        "gerado": extraido,
        "saida_bruta": saida_bruta,
    })

with open('resultados_finetuned.json', 'w', encoding='utf-8') as f:
    json.dump(resultados_finetuned, f, ensure_ascii=False, indent=2)

print("fine-tuning concluído.")

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


fine-tuning concluído.
